# Chest X-ray report generation on IU-Xray

Frontal chest X-ray to a findings paragraph: DenseNet-121, 49 image tokens, an MLP, then DistilGPT-2. A second setup also reads the three most similar training reports.

Run this on a Colab T4 GPU. Set `DATA_SOURCE` in the config cell. Outputs are saved under `MyDrive/cxr_report_gen/`. The test split runs only when `RUN_TEST` is true.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/cxr_report_gen"
DATA = f"{PROJECT}/data"
CKPTS = f"{PROJECT}/checkpoints"
RESULTS = f"{PROJECT}/results"
for d in (DATA, CKPTS, RESULTS):
    os.makedirs(d, exist_ok=True)

!pip install -q torchxrayvision rouge-score nltk kagglehub
!pip install -q f1chexbert || echo "f1chexbert failed: CheXbert F1 will be skipped"


In [ ]:
DATA_SOURCE = "kaggle"
RUN = "full_abn3_ft"
FINETUNE_ENC = True
ABN_WEIGHT = 3.0
EPOCHS, BS = 15, 32
RETRAIN = False
RUN_TEST = False
SEED = 42


In [ ]:
import re, random, numpy as np, pandas as pd
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda"

def have(*names):
    return all(os.path.exists(f"{DATA}/{n}") for n in names)

need = ["studies.csv", "feat_pooled.npy", "feat_spatial.npy"] + (["feat_t3.npy"] if FINETUNE_ENC else [])

if not have(*need):
    if DATA_SOURCE != "kaggle":
        raise FileNotFoundError(
            f"Missing {[n for n in need if not have(n)]} in {DATA}. Upload them to Google Drive, "
            f"or set DATA_SOURCE = 'kaggle' in cell 2 to build them here.")



    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception as e:
        print("Kaggle secrets not found, trying without login:", e)
    import kagglehub
    SRC = kagglehub.dataset_download("raddar/chest-xrays-indiana-university")

    def find_dir(name, root):
        for d, _, fs in os.walk(root):
            if name in fs:
                return d
        raise FileNotFoundError(name)

    CSV_DIR = find_dir("indiana_reports.csv", SRC)
    proj = pd.read_csv(f"{CSV_DIR}/indiana_projections.csv")
    rep = pd.read_csv(f"{CSV_DIR}/indiana_reports.csv")
    IMG_DIR = find_dir(proj.filename.iloc[0], SRC)

    if have("studies.csv"):
        df = pd.read_csv(f"{DATA}/studies.csv")
    else:
        def clean(s):
            s = str(s).lower()
            s = re.sub(r"x{2,}", " ", s)
            s = re.sub(r"([.,;:])", r" \1 ", s)
            s = re.sub(r"[^a-z0-9.,;:\- ]", " ", s)
            return re.sub(r"\s+", " ", s).strip()

        front = proj[proj.projection == "Frontal"].drop_duplicates("uid")
        df = front.merge(rep, on="uid")
        df["target"] = df.findings.fillna("").map(clean)
        df = df[df.target.str.len() > 0].reset_index(drop=True)
        df["normal"] = df.Problems.fillna("").str.strip().str.lower().eq("normal")
        uids = df.uid.unique().copy()
        rng = np.random.default_rng(SEED); rng.shuffle(uids)
        a, b = int(0.7 * len(uids)), int(0.8 * len(uids))
        split = {u: "train" for u in uids[:a]}
        split.update({u: "val" for u in uids[a:b]})
        split.update({u: "test" for u in uids[b:]})
        df["split"] = df.uid.map(split)
        df.to_csv(f"{DATA}/studies.csv", index=False)

    import torchxrayvision as xrv
    import torch.nn.functional as F
    from PIL import Image

    enc = xrv.models.DenseNet(weights="densenet121-res224-all").to(dev).eval()
    crop = xrv.datasets.XRayCenterCrop()
    body, head = enc.features[:-2], enc.features[-2:]

    def load(fn):
        img = np.array(Image.open(f"{IMG_DIR}/{fn}").convert("L"), dtype=np.float32)
        img = crop(xrv.datasets.normalize(img, 255)[None])
        x = torch.from_numpy(img)[None]
        return F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)[0]

    pooled, spatial, t3 = [], [], []
    with torch.no_grad():
        for i in range(0, len(df), 64):
            x = torch.stack([load(f) for f in df.filename[i:i + 64]]).to(dev)
            mid = body(x)
            f = F.relu(head(mid))
            t3.append(mid.half().cpu())
            spatial.append(f.flatten(2).transpose(1, 2).half().cpu())
            pooled.append(f.mean((2, 3)).half().cpu())
            if i % 640 == 0: print(f"{i}/{len(df)}")
    np.save(f"{DATA}/feat_pooled.npy", torch.cat(pooled).numpy())
    np.save(f"{DATA}/feat_spatial.npy", torch.cat(spatial).numpy())
    np.save(f"{DATA}/feat_t3.npy", torch.cat(t3).numpy())
    del enc; torch.cuda.empty_cache()
    print("features saved to Drive")

df = pd.read_csv(f"{DATA}/studies.csv")
df["target"] = df["target"].fillna("").astype(str)
df["normal"] = df["normal"].astype(bool)
tr_idx = np.where(df.split == "train")[0]
va_idx = np.where(df.split == "val")[0]
te_idx = np.where(df.split == "test")[0]
assert len(np.load(f"{DATA}/feat_pooled.npy", mmap_mode="r")) == len(df), "features do not match studies.csv"
print(len(df), "studies |", df.target.nunique(), "distinct reports |", f"{df.normal.mean():.1%} normal")
print({"train": len(tr_idx), "val": len(va_idx), "test": len(te_idx)})


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

P = np.load(f"{DATA}/feat_pooled.npy").astype(np.float32)
y = (~df.normal).astype(int).values
sc = StandardScaler().fit(P[tr_idx])
probe = LogisticRegression(max_iter=3000, C=0.1).fit(sc.transform(P[tr_idx]), y[tr_idx])
auc = roc_auc_score(y[va_idx], probe.predict_proba(sc.transform(P[va_idx]))[:, 1])
print(f"val AUC (abnormal vs normal): {auc:.3f}   (near 0.5 = broken features; ~0.75 = usable signal)")


In [ ]:
import glob, os, shutil
from huggingface_hub import hf_hub_download

target = "/root/.cache/chexbert/chexbert.pth"
os.makedirs(os.path.dirname(target), exist_ok=True)
found = [p for p in glob.glob("/root/.cache/**/chexbert.pth", recursive=True) if p != target]
src = found[0] if found else hf_hub_download("StanfordAIMI/RRG_scorers", "chexbert.pth")
shutil.copy(os.path.realpath(src), target)
print("copied", src, "->", target, round(os.path.getsize(target) / 1e9, 2), "GB")


In [ ]:
import warnings
from nltk.translate.bleu_score import corpus_bleu
from rouge_score import rouge_scorer
warnings.filterwarnings("ignore", message=".*counts of .*-gram overlaps.*")

_rs = rouge_scorer.RougeScorer(["rougeL"])

def bleu4(refs, hyps):
    return corpus_bleu([[r.split()] for r in refs], [h.split() for h in hyps])

def rougeL(refs, hyps):
    return float(np.mean([_rs.score(r, h)["rougeL"].fmeasure for r, h in zip(refs, hyps)]))

chexbert = None
for attempt in range(2):
    try:
        from f1chexbert import F1CheXbert
        chexbert = F1CheXbert(device=dev)
        break
    except Exception as e:
        print(f"CheXbert attempt {attempt + 1} failed:", e)
        os.system("rm -rf /root/.cache/chexbert")

def chex_f1(refs, hyps):

    _, _, _, cr5 = chexbert(hyps=list(hyps), refs=list(refs))
    return cr5["micro avg"]["f1-score"]

print("CheXbert ready" if chexbert else "CheXbert unavailable: do not report clinical metrics from this run")

def evaluate(name, idx, hyps):
    refs, hyps = df.target.values[idx].tolist(), list(hyps)
    nm = df.normal.values[idx]
    ra = [r for r, m in zip(refs, nm) if not m]
    ha = [h for h, m in zip(hyps, nm) if not m]
    row = {"system": name, "n": len(refs), "BLEU-4": bleu4(refs, hyps), "ROUGE-L": rougeL(refs, hyps),
           "distinct": len(set(hyps)), "pct_majority_text": float(np.mean([h == MAJ for h in hyps])),
           "BLEU-4 abnormal": bleu4(ra, ha)}
    if chexbert:
        row["CheXbert F1"] = chex_f1(refs, hyps)
        row["CheXbert F1 abnormal"] = chex_f1(ra, ha)
    return row

def boot_ci(fn, idx, hyps, n=500):
    refs = np.array(df.target.values[idx], dtype=object)
    hyps = np.array(list(hyps), dtype=object)
    g = np.random.default_rng(0)
    vals = [fn(list(refs[s]), list(hyps[s])) for s in (g.integers(0, len(refs), len(refs)) for _ in range(n))]
    return np.round(np.percentile(vals, [2.5, 97.5]), 4).tolist()


In [ ]:
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

def _encode_plus(self, text, *a, **k):

    if isinstance(text, (list, tuple)) and all(isinstance(t, str) for t in text):
        return {"input_ids": [self.cls_token_id] + self.convert_tokens_to_ids(list(text)) + [self.sep_token_id]}
    return self(text, *a, **k)

PreTrainedTokenizerBase.encode_plus = _encode_plus

print(chexbert(hyps=["no pleural effusion ."], refs=["small left pleural effusion ."])[3]["micro avg"])


In [ ]:
MAJ = pd.Series(df.target.values[tr_idx]).value_counts().index[0]
Pn = P / np.linalg.norm(P, axis=1, keepdims=True)

def nn_preds(idx):
    return df.target.values[tr_idx][(Pn[idx] @ Pn[tr_idx].T).argmax(1)]

val_rows = [evaluate("Majority report", va_idx, [MAJ] * len(va_idx)),
            evaluate("Nearest training image", va_idx, nn_preds(va_idx))]
pd.DataFrame(val_rows).round(4)


In [ ]:
import copy
import torch.nn.functional as F
import torchxrayvision as xrv
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, get_cosine_schedule_with_warmup

tok = GPT2TokenizerFast.from_pretrained("distilgpt2")
EOS = tok.eos_token_id
MAXLEN = 128
FEATS = torch.from_numpy(np.load(f"{DATA}/feat_t3.npy" if FINETUNE_ENC else f"{DATA}/feat_spatial.npy"))

class DS(torch.utils.data.Dataset):
    def __init__(self, idx): self.idx = idx
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = int(self.idx[k])
        return i, tok(df.target[i]).input_ids[:MAXLEN - 1] + [EOS]

def collate(batch):
    idx, seqs = zip(*batch)
    L = max(map(len, seqs))
    ids = torch.full((len(batch), L), EOS)
    lab = torch.full((len(batch), L), -100)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for j, s in enumerate(seqs):
        ids[j, :len(s)] = torch.tensor(s); lab[j, :len(s)] = torch.tensor(s); att[j, :len(s)] = 1
    return torch.tensor(idx), ids, lab, att

class Captioner(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.lm = GPT2LMHeadModel.from_pretrained("distilgpt2")
        d = self.lm.config.n_embd
        self.enc_head = None
        if FINETUNE_ENC:
            dn = xrv.models.DenseNet(weights="densenet121-res224-all")
            self.enc_head = copy.deepcopy(dn.features[-2:])
            del dn
        self.map = torch.nn.Sequential(torch.nn.Linear(1024, d), torch.nn.GELU(),
                                       torch.nn.Linear(d, d), torch.nn.LayerNorm(d))

        self.map[-1].weight.data.fill_(self.lm.transformer.wte.weight.std().item())

    def train(self, mode=True):
        super().train(mode)
        if self.enc_head is not None:
            for m in self.enc_head.modules():
                if isinstance(m, torch.nn.BatchNorm2d): m.eval()
        return self

    def tokens(self, feats):
        if self.enc_head is not None:
            feats = F.relu(self.enc_head(feats)).flatten(2).transpose(1, 2)
        return self.map(feats)

    def forward(self, feats, ids, lab, att):
        pre = self.tokens(feats)
        emb = torch.cat([pre, self.lm.transformer.wte(ids)], 1)
        B, P_ = pre.shape[:2]
        att = torch.cat([torch.ones(B, P_, dtype=att.dtype, device=att.device), att], 1)
        lab = torch.cat([torch.full((B, P_), -100, dtype=lab.dtype, device=lab.device), lab], 1)

        return self.lm(inputs_embeds=emb, attention_mask=att, labels=lab).loss

    @torch.no_grad()
    def generate(self, feats, beams=1, ngram=0):
        pre = self.tokens(feats)
        att = torch.ones(pre.shape[:2], dtype=torch.long, device=pre.device)
        out = self.lm.generate(inputs_embeds=pre, attention_mask=att, max_new_tokens=MAXLEN,
                               num_beams=beams, do_sample=False, no_repeat_ngram_size=ngram,
                               eos_token_id=EOS, pad_token_id=EOS)
        return [tok.decode(o, skip_special_tokens=True).strip() for o in out]

def to_dev(idx, ids, lab, att):
    return FEATS[idx].to(dev).float(), ids.to(dev), lab.to(dev), att.to(dev)

def model_preds(idx, beams=1, ngram=0):
    model.eval(); out = []
    for i in range(0, len(idx), 64):
        out += model.generate(FEATS[torch.as_tensor(idx[i:i + 64])].to(dev).float(), beams, ngram)
    return out


In [ ]:
_, ids, lab, att = collate([DS(tr_idx)[0]])
assert lab[0, 0].item() == tok(df.target[int(tr_idx[0])]).input_ids[0], "first word must be a target"
assert lab[0, att[0].sum() - 1].item() == EOS, "EOS must be a target"
assert (lab[0][att[0] == 0] == -100).all(), "only padding is ignored"
print("label tests passed; decoding starts from the image tokens, not from EOS")


In [ ]:
import json, shutil, time

K_RAG = 3
CTX_TOKENS = 60
SEEDS = [1, 2, 3]
CONFIGS = {"norag": 0, f"rag{K_RAG}": K_RAG}
FINETUNE_ENC = True
FEATS = torch.from_numpy(np.load(f"{DATA}/feat_t3.npy"))

sims = Pn @ Pn[tr_idx].T
pos_in_train = {int(t): j for j, t in enumerate(tr_idx)}
for i, j in pos_in_train.items():
    sims[i, j] = -np.inf
NBR = tr_idx[np.argsort(-sims, axis=1)[:, :K_RAG]]
assert not np.isin(NBR, np.concatenate([va_idx, te_idx])).any(), "retrieval must use training studies only"
assert all(i not in NBR[i] for i in tr_idx), "a training study must not retrieve its own report"



SEP = tok(" |").input_ids
PROMPT = tok(" findings:").input_ids

def ctx_ids(i):
    out = []
    for n in NBR[i]:
        r = tok(df.target[int(n)]).input_ids[:CTX_TOKENS]
        out += r + [EOS] * (CTX_TOKENS - len(r)) + SEP
    return out + PROMPT

CTX = torch.tensor([ctx_ids(i) for i in range(len(df))])
print("context tokens per study:", CTX.shape[1], "| example retrieved report:", df.target[int(NBR[va_idx[0], 0])][:120])


In [ ]:
import gc
for v in ("m", "model"):
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()

MICRO, ACCUM = 16, 2

@torch.no_grad()
def rag_val_loss(m):
    m.eval(); tot = n = 0
    for batch in torch.utils.data.DataLoader(DS(va_idx), batch_size=16, collate_fn=collate):
        f, c, ids, lab, att = batch_to_dev(*batch)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            loss = m(f, c, ids, lab, att)
        k = (lab != -100).sum().item(); tot += loss.item() * k; n += k
    return tot / n

def rag_preds(m, idx, beams=1, ngram=0):
    m.eval(); out = []
    for i in range(0, len(idx), 16):
        b = torch.as_tensor(idx[i:i + 16])
        out += m.generate(FEATS[b].to(dev).float(), CTX[b].to(dev), beams, ngram)
    return out

def train_run(name, k, seed):
    ckpt = f"{CKPTS}/{name}.pt"
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    m = RagCaptioner(k).to(dev)
    if os.path.exists(ckpt):
        m.load_state_dict(torch.load(ckpt, map_location=dev))
        print(f"{name}: checkpoint exists, skipping training")
        return m
    w = np.where(df.normal.values[tr_idx], 1.0, ABN_WEIGHT)
    g = torch.Generator().manual_seed(seed)
    sampler = torch.utils.data.WeightedRandomSampler(torch.tensor(w), len(tr_idx), replacement=True, generator=g)
    dl = torch.utils.data.DataLoader(DS(tr_idx), batch_size=MICRO, sampler=sampler, collate_fn=collate)
    steps = (len(dl) + ACCUM - 1) // ACCUM
    groups = [{"params": m.map.parameters(), "lr": 5e-4}, {"params": m.lm.parameters(), "lr": 5e-5},
              {"params": m.enc_head.parameters(), "lr": 1e-5}]
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    sched = get_cosine_schedule_with_warmup(opt, steps, EPOCHS * steps)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, t0 = float("inf"), time.time()
    for ep in range(1, EPOCHS + 1):
        m.train(); run = 0; opt.zero_grad()
        for j, batch in enumerate(dl, 1):
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                loss = m(*batch_to_dev(*batch))


In [ ]:
class RagCaptioner(Captioner):
    def __init__(self, k):
        super().__init__()
        self.k = k

    def prefix(self, feats, ctx):
        pre = self.tokens(feats)
        if self.k:
            pre = torch.cat([pre, self.lm.transformer.wte(ctx)], 1)
        return pre

    def forward(self, feats, ctx, ids, lab, att):
        pre = self.prefix(feats, ctx)
        emb = torch.cat([pre, self.lm.transformer.wte(ids)], 1)
        B, P_ = pre.shape[:2]
        att = torch.cat([torch.ones(B, P_, dtype=att.dtype, device=att.device), att], 1)
        lab = torch.cat([torch.full((B, P_), -100, dtype=lab.dtype, device=lab.device), lab], 1)
        return self.lm(inputs_embeds=emb, attention_mask=att, labels=lab).loss

    @torch.no_grad()
    def generate(self, feats, ctx, beams=1, ngram=0):
        pre = self.prefix(feats, ctx)
        att = torch.ones(pre.shape[:2], dtype=torch.long, device=pre.device)
        out = self.lm.generate(inputs_embeds=pre, attention_mask=att, max_new_tokens=MAXLEN,
                               num_beams=beams, do_sample=False, no_repeat_ngram_size=ngram,
                               eos_token_id=EOS, pad_token_id=EOS)
        return [tok.decode(o, skip_special_tokens=True).strip() for o in out]

AMP = dev == "cuda"

def batch_to_dev(idx, ids, lab, att):
    return FEATS[idx].to(dev).float(), CTX[idx].to(dev), ids.to(dev), lab.to(dev), att.to(dev)

@torch.no_grad()
def rag_val_loss(m):
    m.eval(); tot = n = 0
    for batch in torch.utils.data.DataLoader(DS(va_idx), batch_size=64, collate_fn=collate):
        f, c, ids, lab, att = batch_to_dev(*batch)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            loss = m(f, c, ids, lab, att)
        k = (lab != -100).sum().item(); tot += loss.item() * k; n += k
    return tot / n

def rag_preds(m, idx, beams=1, ngram=0):
    m.eval(); out = []
    for i in range(0, len(idx), 64):
        b = torch.as_tensor(idx[i:i + 64])
        out += m.generate(FEATS[b].to(dev).float(), CTX[b].to(dev), beams, ngram)
    return out

def train_run(name, k, seed):
    ckpt = f"{CKPTS}/{name}.pt"
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    m = RagCaptioner(k).to(dev)
    if os.path.exists(ckpt):
        m.load_state_dict(torch.load(ckpt, map_location=dev))
        print(f"{name}: checkpoint exists, skipping training")
        return m
    w = np.where(df.normal.values[tr_idx], 1.0, ABN_WEIGHT)
    g = torch.Generator().manual_seed(seed)
    sampler = torch.utils.data.WeightedRandomSampler(torch.tensor(w), len(tr_idx), replacement=True, generator=g)
    dl = torch.utils.data.DataLoader(DS(tr_idx), batch_size=BS, sampler=sampler, collate_fn=collate)
    groups = [{"params": m.map.parameters(), "lr": 5e-4}, {"params": m.lm.parameters(), "lr": 5e-5},
              {"params": m.enc_head.parameters(), "lr": 1e-5}]
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    sched = get_cosine_schedule_with_warmup(opt, len(dl), EPOCHS * len(dl))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, t0 = float("inf"), time.time()
    for ep in range(1, EPOCHS + 1):
        m.train(); run = 0
        for batch in dl:
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                loss = m(*batch_to_dev(*batch))
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        vl = rag_val_loss(m)
        print(f"{name} ep {ep:2d} | train {run / len(dl):.3f} | val {vl:.3f} | {(time.time() - t0) / 60:.1f} min")
        if vl < best:
            best = vl
            torch.save(m.state_dict(), "/content/tmp.pt")
            shutil.move("/content/tmp.pt", ckpt)
    m.load_state_dict(torch.load(ckpt, map_location=dev))
    return m

print("ready:", list(CONFIGS), "x seeds", SEEDS)


In [ ]:
assert chexbert, "Run the CheXbert fix cells first: model choice uses CheXbert F1"
VAL_LOG = f"{RESULTS}/seed_runs_val.csv"
done = pd.read_csv(VAL_LOG) if os.path.exists(VAL_LOG) else pd.DataFrame(columns=["run"])

for family, k in CONFIGS.items():
    for seed in SEEDS:
        name = f"{family}_s{seed}"
        if name in set(done.run):
            print(f"{name}: already validated, skipping"); continue
        m = train_run(name, k, seed)
        best = None
        for beams, ngram in [(1, 0), (1, 4), (3, 4)]:
            r = evaluate(name, va_idx, rag_preds(m, va_idx, beams, ngram))
            if best is None or r["CheXbert F1"] > best["CheXbert F1"]:
                best = {**r, "beams": beams, "ngram": ngram}
        best.update(run=name, family=family, seed=seed, k=k, val_loss=rag_val_loss(m))
        done = pd.concat([done[done.run != name], pd.DataFrame([best])], ignore_index=True)
        done.to_csv(VAL_LOG, index=False)
        print(f"{name}: CheXbert F1 {best['CheXbert F1']:.4f} | abnormal {best['CheXbert F1 abnormal']:.4f} "
              f"| BLEU-4 {best['BLEU-4']:.4f} | decoding beams={best['beams']} ngram={best['ngram']}")
        del m; torch.cuda.empty_cache()


In [ ]:
METRICS = ["CheXbert F1", "CheXbert F1 abnormal", "BLEU-4", "ROUGE-L", "distinct"]

def summarize(log, base_rows):
    rows = [{"system": b["system"], "seeds": 1, **{c: f"{b[c]:.4f}" for c in METRICS}} for b in base_rows]
    for fam, grp in log.groupby("family", sort=False):
        rows.append({"system": fam, "seeds": len(grp),
                     **{c: f"{grp[c].mean():.4f} ± {grp[c].std(ddof=1) if len(grp) > 1 else 0:.4f}" for c in METRICS}})
    return pd.DataFrame(rows)

val_log = pd.read_csv(VAL_LOG)
val_base = [evaluate("Majority report", va_idx, [MAJ] * len(va_idx)),
            evaluate("Nearest training image", va_idx, nn_preds(va_idx))]
val_summary = summarize(val_log, val_base)
val_summary.to_csv(f"{RESULTS}/seed_summary_val.csv", index=False)
display(val_summary)
display(val_log[["run", "CheXbert F1", "CheXbert F1 abnormal", "BLEU-4", "distinct", "beams", "ngram"]].round(4))


In [ ]:
m0 = train_run(f"norag_s{SEEDS[0]}", 0, SEEDS[0]); m1 = train_run(f"rag{K_RAG}_s{SEEDS[0]}", K_RAG, SEEDS[0])
d0 = val_log.set_index("run").loc[f"norag_s{SEEDS[0]}"]; d1 = val_log.set_index("run").loc[f"rag{K_RAG}_s{SEEDS[0]}"]
ab = va_idx[~df.normal.values[va_idx]][:10]
g0 = rag_preds(m0, ab, int(d0.beams), int(d0.ngram)); g1 = rag_preds(m1, ab, int(d1.beams), int(d1.ngram))
for i, a, b, c in zip(ab, g0, g1, nn_preds(ab)):
    print("REF   :", df.target[i]); print("NO RAG:", a); print("RAG   :", b); print("NN    :", c); print("-" * 80)
del m0, m1; torch.cuda.empty_cache()


In [ ]:
RUN_SEED_TEST = True
TEST_LOG = f"{RESULTS}/seed_runs_test.csv"
if not RUN_SEED_TEST:
    print("RUN_SEED_TEST is False. Set it to True only when you are done with validation.")
elif os.path.exists(TEST_LOG):
    print(f"{TEST_LOG} exists: test was already run.")
    display(pd.read_csv(f"{RESULTS}/seed_summary_test.csv"))
else:
    rows = []
    for _, r in pd.read_csv(VAL_LOG).iterrows():
        m = train_run(r.run, int(r.k), int(r.seed))
        hyps = rag_preds(m, te_idx, int(r.beams), int(r.ngram))
        rows.append({**evaluate(r.run, te_idx, hyps), "run": r.run, "family": r.family, "seed": r.seed})
        pd.DataFrame({"uid": df.uid.values[te_idx], "reference": df.target.values[te_idx], "generated": hyps}
                     ).to_csv(f"{RESULTS}/test_predictions_{r.run}.csv", index=False)
        del m; torch.cuda.empty_cache()
    test_log = pd.DataFrame(rows); test_log.to_csv(TEST_LOG, index=False)
    test_base = [evaluate("Majority report", te_idx, [MAJ] * len(te_idx)),
                 evaluate("Nearest training image", te_idx, nn_preds(te_idx))]
    test_summary = summarize(test_log, test_base)
    test_summary.to_csv(f"{RESULTS}/seed_summary_test.csv", index=False)
    display(test_summary)


In [ ]:
import shutil
w = np.where(df.normal.values[tr_idx], 1.0, ABN_WEIGHT)
sampler = torch.utils.data.WeightedRandomSampler(torch.tensor(w), len(tr_idx), replacement=True)
dl_tr = torch.utils.data.DataLoader(DS(tr_idx), batch_size=BS, sampler=sampler, collate_fn=collate)
dl_va = torch.utils.data.DataLoader(DS(va_idx), batch_size=64, collate_fn=collate)

model = Captioner().to(dev)
groups = [{"params": model.map.parameters(), "lr": 5e-4}, {"params": model.lm.parameters(), "lr": 5e-5}]
if model.enc_head is not None:
    groups.append({"params": model.enc_head.parameters(), "lr": 1e-5})
opt = torch.optim.AdamW(groups, weight_decay=0.01)
sched = get_cosine_schedule_with_warmup(opt, len(dl_tr), EPOCHS * len(dl_tr))

@torch.no_grad()
def val_loss(mode="real"):
    model.eval(); tot = n = 0
    for batch in dl_va:
        f, ids, lab, att = to_dev(*batch)
        if mode == "zero": f = torch.zeros_like(f)
        if mode == "shuffle": f = f[torch.randperm(len(f), device=f.device)]
        k = (lab != -100).sum().item()
        tot += model(f, ids, lab, att).item() * k; n += k
    return tot / n

CKPT = f"{CKPTS}/{RUN}.pt"
if os.path.exists(CKPT) and not RETRAIN:
    print("checkpoint exists, skipping training:", CKPT)
else:
    print(f"{RUN} | train {len(tr_idx)} | epochs {EPOCHS} | finetune encoder {FINETUNE_ENC} | abnormal weight {ABN_WEIGHT}")
    best, history = float("inf"), []
    for ep in range(1, EPOCHS + 1):
        model.train(); run = 0
        for batch in dl_tr:
            loss = model(*to_dev(*batch))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); run += loss.item()
        vl = val_loss()
        distinct = len(set(model_preds(va_idx[:128])))
        history.append({"epoch": ep, "train_loss": run / len(dl_tr), "val_loss": vl, "distinct_128": distinct})
        print(f"ep {ep:2d} | train {run / len(dl_tr):.3f} | val {vl:.3f} | distinct on 128 val: {distinct}")
        if vl < best:
            best = vl
            torch.save(model.state_dict(), "/content/tmp.pt")
            shutil.move("/content/tmp.pt", CKPT)
    pd.DataFrame(history).to_csv(f"{CKPTS}/{RUN}.history.csv", index=False)

model.load_state_dict(torch.load(CKPT, map_location=dev))
print("loaded", CKPT, "| val loss:", round(val_loss(), 4))


In [ ]:
for mode in ("real", "zero", "shuffle"):
    print(f"{mode:8s} val loss: {val_loss(mode):.4f}")
print("real clearly lower than zero/shuffle = image is used; all equal = image ignored")


In [ ]:
import json
settings = [(1, 0), (1, 4), (3, 0), (3, 4)]
rows = []
for beams, ngram in settings:
    r = evaluate(f"Model beams={beams} ngram={ngram}", va_idx, model_preds(va_idx, beams, ngram))
    r["beams"], r["ngram"] = beams, ngram
    rows.append(r)
tune = pd.DataFrame(rows)
key = "CheXbert F1" if chexbert else "ROUGE-L"
best_row = tune.sort_values(key, ascending=False).iloc[0]
BEST_DEC = (int(best_row["beams"]), int(best_row["ngram"]))
json.dump({"beams": BEST_DEC[0], "ngram": BEST_DEC[1], "chosen_by": key}, open(f"{CKPTS}/{RUN}.decode.json", "w"))
print("chosen on validation by", key, "->", BEST_DEC)

LOG = f"{RESULTS}/val_log.csv"
entry = best_row.drop(["beams", "ngram"]).to_dict()
entry.update(run=RUN, finetune_enc=FINETUNE_ENC, abn_weight=ABN_WEIGHT, epochs=EPOCHS,
             beams=BEST_DEC[0], ngram=BEST_DEC[1], val_loss=round(val_loss(), 4))
log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame(columns=["run"])
log = pd.concat([log[log.run != RUN], pd.DataFrame([entry])], ignore_index=True)
log.to_csv(LOG, index=False)

display(pd.concat([pd.DataFrame(val_rows), tune.drop(columns=["beams", "ngram"])]).round(4))
print("All runs so far:")
display(log.round(4))


In [ ]:
ab = va_idx[~df.normal.values[va_idx]][:15]
for i, g, nb in zip(ab, model_preds(ab, *BEST_DEC), nn_preds(ab)):
    print("REF:", df.target[i]); print("GEN:", g); print("NN :", nb); print("-" * 80)


In [ ]:
OUT = f"{RESULTS}/test_{RUN}.csv"
if not RUN_TEST:
    print("RUN_TEST is False. Set it to True in cell 2 only when every choice above is final.")
elif os.path.exists(OUT):
    print(f"{OUT} exists: this checkpoint was already tested. Test runs once.")
else:
    systems = {"Majority report": [MAJ] * len(te_idx),
               "Nearest training image": list(nn_preds(te_idx)),
               "Model": model_preds(te_idx, *BEST_DEC)}
    out = []
    for name, h in systems.items():
        r = evaluate(name, te_idx, h)
        r["BLEU-4 95% CI"] = boot_ci(bleu4, te_idx, h)
        r["ROUGE-L 95% CI"] = boot_ci(rougeL, te_idx, h)
        out.append(r)
    res = pd.DataFrame(out)
    res.to_csv(OUT, index=False)
    pd.DataFrame({"uid": df.uid.values[te_idx], "reference": df.target.values[te_idx], **systems}).to_csv(
        f"{RESULTS}/test_predictions_{RUN}.csv", index=False)
    display(res.round(4))


In [ ]:
!pip install -q gradio
import glob
import gradio as gr
import torch.nn.functional as F
import torchxrayvision as xrv
from PIL import Image

FINETUNE_ENC = True
enc_full = xrv.models.DenseNet(weights="densenet121-res224-all").to(dev).eval()
enc_body, enc_top = enc_full.features[:-2], enc_full.features[-2:]
crop = xrv.datasets.XRayCenterCrop()


vlog = pd.read_csv(f"{RESULTS}/seed_runs_val.csv").set_index("run")
MODELS = {}
for label, fam in [("No RAG (best model)", "norag"), (f"RAG ({K_RAG} retrieved reports)", f"rag{K_RAG}")]:
    run = vlog[vlog.family == fam]["CheXbert F1"].idxmax()
    k, beams, ngram = int(vlog.loc[run, "k"]), int(vlog.loc[run, "beams"]), int(vlog.loc[run, "ngram"])
    m = RagCaptioner(k).to(dev).eval()
    m.load_state_dict(torch.load(f"{CKPTS}/{run}.pt", map_location=dev))
    MODELS[label] = (m, beams, ngram)
    print(f"loaded {label}: {run} (val CheXbert F1 {vlog.loc[run, 'CheXbert F1']:.3f})")

def preprocess(pil):
    """Same steps as training: grayscale, scale to [-1024, 1024], centre crop, 224x224."""
    a = np.array(pil).astype(np.float32)
    if a.ndim == 3:
        a = a[..., :3].mean(-1)
    if a.max() > 255:
        a = a / a.max() * 255
    a = crop(xrv.datasets.normalize(a, 255)[None])
    x = torch.from_numpy(a)[None]
    return F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False).to(dev)

@torch.no_grad()
def report(pil, choice):
    if pil is None:
        return "Upload a frontal chest X-ray.", ""
    x = preprocess(pil)
    mid = enc_body(x)
    pooled = F.relu(enc_top(mid)).mean((2, 3))[0].float().cpu().numpy()
    pooled /= np.linalg.norm(pooled)
    sims = Pn[tr_idx] @ pooled
    pos = np.argsort(-sims)[:K_RAG]
    ctx = []
    for nb in tr_idx[pos]:
        r = tok(df.target[int(nb)]).input_ids[:CTX_TOKENS]
        ctx += r + [EOS] * (CTX_TOKENS - len(r)) + SEP
    ctx = torch.tensor([ctx + PROMPT], device=dev)
    m, beams, ngram = MODELS[choice]
    text = m.generate(mid.float(), ctx, beams, ngram)[0]
    similar = "\n\n".join(f"{j + 1}. similarity {sims[p]:.2f}: {df.target[int(tr_idx[p])]}" for j, p in enumerate(pos))
    return text, similar

def predict(path, choice=None):
    """Script use: predict('/content/my_xray.png')"""
    return report(Image.open(path), choice or list(MODELS)[0])[0]

ex_dir = glob.glob("/root/.cache/kagglehub/**/images_normalized", recursive=True)
examples = [[f"{ex_dir[0]}/{fn}", list(MODELS)[0]] for fn in df.filename.values[te_idx[:3]]] if ex_dir else None

demo = gr.Interface(
    fn=report,
    inputs=[gr.Image(type="pil", label="Frontal chest X-ray (PNG or JPG)"),
            gr.Radio(list(MODELS), value=list(MODELS)[0], label="Model")],
    outputs=[gr.Textbox(label="Generated findings", lines=6),
             gr.Textbox(label="Most similar training reports (what RAG retrieves)", lines=10)],
    title="Chest X-ray report generation: IU-Xray research demo",
    description="Research prototype, not for clinical use. Test CheXbert F1 is 0.32, so it misses many findings.",
    examples=examples,
)
demo.launch(debug=False)
